# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library. The dataset includes comprehensive mass spectrometry analyses of extracellular vesicles isolated from human mast cells, featuring protein abundance, modifications, sequences, and related fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', 'N/A'))
print("Identifier:", getattr(metadata, 'identifier', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List available record sets with their @id and fields
record_set_ids = []
print("\nAvailable Record Sets in the Dataset:")
for rs in metadata.record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (@id: {f.id}, dataType: {f.data_type})")
    print()
# Show an example of one record from each RecordSet
ds = dataset
print("\n----- Example record from each RecordSet -----")
for record_set_id in record_set_ids:
    print(f"\nFirst record in RecordSet {record_set_id}:")
    sample = next(ds.records(record_set=record_set_id), None)
    if sample:
        print(sample)
    else:
        print("No records found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always use the record set and field `@id`s listed in the overview.

If you are unsure which record set contains the main tabular data (usually proteins/observations), check the output above and select the appropriate `@id`.

In [ ]:
# Let's load all record sets into dataframes using their @id fields
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet {record_set_id} with columns: {df.columns.tolist()}")
        else:
            print(f"RecordSet {record_set_id} has no records.")
    except Exception as e:
        print(f"Error loading RecordSet {record_set_id}: {e}")

# Choose the primary record set for downstream analysis. Edit the record_set_id below as appropriate:
# (For example, for this dataset, it's often a table-like 'Protein Table' record set)
primary_record_set_id = record_set_ids[0]  # change index as appropriate if multiple sets exist
df = dataframes[primary_record_set_id]

print(f"\nAvailable columns in {primary_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps using field `@id`s.
- Filter numeric fields (e.g., a quantitation or counts field)
- Normalize across records
- Group by key attributes (such as experimental condition or protein family if available)

Replace `numeric_field_id` and `group_field_id` with relevant field `@id`s from previous steps.

In [ ]:
# Example EDA: filtering, normalization, grouping using field @id's
# Select suitable numeric field and group field from previous code output
numeric_field_id = df.select_dtypes(include=[float, int]).columns[0] if len(df.select_dtypes(include=[float, int]).columns) > 0 else None
group_field_id = None
# Try to guess a groupable field (object type, but not unique per row)
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() < df.shape[0] and col != numeric_field_id:
        group_field_id = col
        break

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean()  # example: filter by mean value
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable group field (@id) found for grouping.")
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. We use matplotlib and seaborn for plotting, referencing fields by their @id.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna())
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- Loaded and inspected metadata & records from the FAIR^2 mass spectrometry dataset using `mlcroissant`.
- Listed available record sets and their field `@id`s for robust programmatic reference.
- Demonstrated extraction, filtering, normalization, grouping, and visualization of data using only Croissant `@id`s.

**This notebook provides a reusable template for programmatic, standards-based FAIR dataset exploration.**

For further analysis, see the dataset documentation and Croissant field/entity definitions.